# Heat pump and electric boiler capacity in GKFX

Exogenous end-of-year capacity (MW) of heat pumps (`GNR_HP*`) and electric boilers (`GNR_BO_ELEC*`; `GNR_EH_HEAT` is a waste-heat source and is excluded) from `base/data/GKFX.inc`, per country and per zone.

In [ ]:
# Parse GKFX.inc: TABLE GKFX1(AAA,GGG,YYY), rows are "area . technology v_2012 ... v_2050"
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

GKFX_PATH = Path("base/data/GKFX.inc")

lines = GKFX_PATH.read_text().splitlines()

# Year header: the first line made of 4-digit years (anything after "/*" is a comment).
# GAMS TABLE entries belong to the column they sit under and blanks mean 0, so values
# are assigned to years by character position, not by token order.
header_idx = next(i for i, l in enumerate(lines) if re.match(r"^\s+\d{4}\b", l))
header = lines[header_idx].split("/*")[0]
year_ends = [(int(m.group()), m.end()) for m in re.finditer(r"\d{4}", header)]
years = [y for y, _ in year_ends]

row_re = re.compile(r"^\s*(\S+)\s*\.\s*(\S+)\s")
rows = []
for line in lines[header_idx + 1:]:
    if line.strip() == ";":
        break
    if line.lstrip().startswith("*"):
        continue
    m = row_re.match(line)
    if m:
        area, tech = m.groups()
        values = dict.fromkeys(years, 0.0)
        for tok in re.finditer(r"\S+", line[m.end():]):
            end = m.end() + tok.end()
            year = min(year_ends, key=lambda ye: abs(ye[1] - end))[0]  # right-aligned under its year
            values[year] = float(tok.group())
        rows.append([area, tech] + list(values.values()))

gkfx = pd.DataFrame(rows, columns=["area", "tech"] + years)

# Keep heat pumps and electric boilers, add country (area prefix) and type
gkfx["type"] = gkfx["tech"].map(
    lambda t: "Heat pump" if t.startswith("GNR_HP") else "Electric boiler" if t.startswith("GNR_BO_ELEC") else None
)
gkfx = gkfx.dropna(subset=["type"])
gkfx["country"] = gkfx["area"].str.extract(r"^([A-Z]{2})")[0]

long = gkfx.melt(id_vars=["area", "country", "tech", "type"], var_name="year", value_name="MW")
by_country = long.groupby(["country", "type", "year"], as_index=False)["MW"].sum()
by_zone = long.groupby(["area", "country", "type", "year"], as_index=False)["MW"].sum()

by_country.pivot_table(index="country", columns=["type", "year"], values="MW").loc[:, (slice(None), [2025, 2030, 2040, 2050])].round(0)


In [ ]:
# Capacity per country over time, one panel per country (shared year axis, free y axis)
COLORS = {"Heat pump": "#2a78d6", "Electric boiler": "#eb6834"}  # series 1 and 2 of the dataviz palette

countries = sorted(by_country["country"].unique())
ncols = 3
nrows = -(-len(countries) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 2.6 * nrows), sharex=True, squeeze=False)

for ax, country in zip(axes.flat, countries):
    d = by_country[by_country["country"] == country]
    for t, color in COLORS.items():
        s = d[d["type"] == t].set_index("year")["MW"]
        ax.plot(s.index, s.values, color=color, lw=2, label=t)
    ax.set_title(country, loc="left", fontsize=10)
    ax.grid(axis="y", lw=0.5, alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes.flat[len(countries):]:
    ax.set_visible(False)

axes[0, 0].legend(frameon=False, fontsize=8)
fig.supylabel("Capacity (MW)")
fig.suptitle("Exogenous capacity in GKFX: heat pumps and electric boilers per country", x=0.01, ha="left")
fig.tight_layout()
plt.show()


In [ ]:
# Capacity per zone in a chosen year (zones = the AAA areas of GKFX)
YEAR = 2025

d = (by_zone[by_zone["year"] == YEAR]
     .pivot_table(index="area", columns="type", values="MW", fill_value=0)
     .reindex(columns=list(COLORS)).fillna(0))
d = d[d.sum(axis=1) > 0].sort_index(ascending=False)

fig, ax = plt.subplots(figsize=(7, 0.28 * len(d) + 1.2))
ypos = range(len(d))
h = 0.38
ax.barh([y + h / 2 for y in ypos], d["Heat pump"], height=h, color=COLORS["Heat pump"], label="Heat pump")
ax.barh([y - h / 2 for y in ypos], d["Electric boiler"], height=h, color=COLORS["Electric boiler"], label="Electric boiler")
ax.set_yticks(list(ypos), d.index, fontsize=8)
ax.set_xlabel("Capacity (MW)")
ax.set_title(f"Capacity per zone in {YEAR}", loc="left", fontsize=10)
ax.grid(axis="x", lw=0.5, alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()
